# Texas Inpatient PUDF: delivery cohort and severe maternal morbidity (2019 and 2020)

This notebook reads the free Texas Inpatient Public Use Data File (THCIC, DSHS) for all four quarters of 2019 and 2020. It finds the delivery hospitalizations and flags severe maternal morbidity (SMM) with the CDC ICD-10 code list. It saves one combined file.

I wrote it without being able to open the data, so it checks the column names before it does anything and stops with a clear message if something does not match the User Manual.

How it works:
- Step 1 to 3: find the zip files, read the column names, map them to my names.
- Step 4: the delivery and SMM code lists. Step 4b: the comorbidity code lists.
- Step 5 and 6: process each quarter in chunks and combine.
- Step 6 also cleans age, race and ethnicity, payer, and adds the death and obstetric unit flags, residence, Public Health Region, admission type and source, and hospital teaching status.
- Step 7 and 8: checks and plots. Step 7b: checks for the comorbidity and new fields.
- Step 9: SMM by race and ethnicity, pooled and 2019 only.
- Step 10: encode the features for modeling and save a second file.

## What to download first

The Data Use Agreement has to be accepted by a person on the DSHS page. Click Agree there, then download the files below by hand. The notebook does not download anything.

Put all the zip files in one folder. Do not rename them. Keep the quarter name, for example 3Q2019, in the file name.

Data files (tab-delimited, from the 2019 and 2020 tables on the PUDF page): PUDF1Q2019 to PUDF4Q2019 and PUDF1Q2020 to PUDF4Q2020, tab-delimited zips.

Manuals (data dictionaries), one per quarter: InpatientDataDictionary1Q2019 to 4Q2019 and 1Q2020 to 4Q2020, PDF. You need at least one 2019 and one 2020 manual open while you run Step 2.

2020 update files (Excel and Word):
- THCICListofRecordIDsofOutpatientdataappearinginQ22020InpatientDataset.xlsx (Q2 2020). These record IDs are outpatient records that sit in the inpatient file and should be removed.
- THCICPUDF3Q2020RecordIDs06-23-2021Addenda.xlsx and .docx (Q3 2020). The Word file says Table3 lists 4 erroneous inpatient records to delete, so the notebook removes those. Table1 and Table2 are inpatient records that were filed in the outpatient file; adding them back needs the outpatient PUDF, which this project does not use (151 records statewide).
- TechnicalUpdate-AddendumTHCICInpatient2Q2020.docx (Q2 2020). Read it too.

No update files are listed for 2019, 1Q2020 or 4Q2020.

Data use rules: do not share the data or upload it to GitHub, and cite DSHS in any write-up.

In [ ]:
import re, io, zipfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=DeprecationWarning)

# ---------------- settings you may change ----------------
dataDir = Path("/Users/sandra/Documents/DataSets")   # folder with the 8 tab-delimited zips and the 2020 Excel file
outDir = dataDir / "output"
years = [2019, 2020]
quarters = [1, 2, 3, 4]
chunkSize = 100_000

# 2020 update files: (Excel file, sheet) with record IDs to remove. Put the Excel files in dataDir.
excludeIdFiles = {
    (2020, 2): (dataDir / "THCICListofRecordIDsofOutpatientdataappearinginQ22020InpatientDataset.xlsx", 0),  # outpatient records in the inpatient file
    (2020, 3): (dataDir / "THCICPUDF3Q2020RecordIDs06-23-2021Addenda.xlsx", "Table3"),                      # 4 erroneous inpatient records
}

# Code tables from the User Manual (same in all 8 quarters, 1Q2019 to 4Q2020)
raceLabels = {"1": "American Indian/Eskimo/Aleut", "2": "Asian or Pacific Islander", "3": "Black", "4": "White", "5": "Other"}
ethnicityLabels = {"1": "Hispanic", "2": "Not Hispanic"}

attBlue, attOrange, attGray = "#009FDB", "#FF8300", "#5A6D7B"
outDir.mkdir(parents=True, exist_ok=True)
print("dataDir exists:", dataDir.exists())

## Step 1. Find the zip files

In [ ]:
def findZip(year, q):
    hits = [p for p in dataDir.glob("*.zip")
            if f"{q}q{year}" in p.name.lower().replace("_", "") and "tab" in p.name.lower()]
    return hits[0] if hits else None

zipFiles = {}
for y in years:
    for q in quarters:
        p = findZip(y, q)
        zipFiles[(y, q)] = p
        print(f"{q}Q{y}:", p.name if p else "MISSING")
missing = [k for k, v in zipFiles.items() if v is None]
assert not missing, f"Missing tab-delimited zip files for: {missing}. Check dataDir and the file names."

## Step 2. Look inside one zip and read the column names
Each zip holds several files (base, charges, classification, facility type). I need the base file. This cell prints every file name so you can check that I picked the right one. If the wrong file is chosen, set baseNameOverride.

In [ ]:
baseNameOverride = None   # example: "IP_PUDF_BASE1_1Q2019_tab.txt"

def listMembers(zipPath):
    with zipfile.ZipFile(zipPath) as z:
        return [(i.filename, i.file_size) for i in z.infolist() if not i.is_dir()]

def pickBase(zipPath):
    members = listMembers(zipPath)
    if baseNameOverride:
        return baseNameOverride
    # skip macOS metadata files (__MACOSX/._name) that sit next to the real files in the zip
    real = [m for m, s in members if "__macosx" not in m.lower() and not Path(m).name.startswith("._")]
    # BASE1 has the diagnoses, procedures, DRG and patient fields. BASE2 only has charges, occurrence and
    # value codes (different columns, joined on RECORD_ID), and this notebook does not need it.
    cands = [m for m in real if "base1" in Path(m).name.lower() and m.lower().endswith((".txt", ".tsv", ".csv", ".dat"))]
    assert len(cands) == 1, f"Expected one BASE1 file in {zipPath.name}, found {cands}. Members: {real}"
    return cands[0]

for m, s in listMembers(zipFiles[(2019, 1)]):
    print(f"{s/1e6:8.1f} MB  {m}")
print()
print("Base file(s) picked for 1Q2019:", pickBase(zipFiles[(2019, 1)]))

In [ ]:
def readHeader(zipPath, member):
    with zipfile.ZipFile(zipPath) as z, z.open(member) as f:
        return pd.read_csv(f, sep="\t", nrows=0, dtype=str, encoding="latin-1").columns.tolist()

columnPatterns = {   # my name -> regexes tried against the upper-case column names, first hit wins
    "recordId":  [r"^RECORD_?ID$"],
    "hospId":    [r"^THCIC_?ID$"],
    "age":       [r"^PAT_AGE$", r"^AGE$"],
    "sex":       [r"^SEX_CODE$", r"^SEX$"],
    "race":      [r"^RACE$"],
    "ethnicity": [r"^ETHNICITY$"],
    "county":    [r"^PAT_COUNTY$", r"COUNTY"],
    "state":     [r"^PAT_STATE$"],
    "zip":       [r"^PAT_ZIP$"],
    "payer":     [r"^FIRST_PAYMENT_SRC$", r"^PRINC_?PAYER", r"SOURCE_OF_PAYMENT", r"^PAYER", r"^EXPECTED"],
    "status":    [r"^PAT_STATUS$", r"PATIENT_STATUS", r"DISCHARGE_STATUS"],
    "los":       [r"^LENGTH_OF_STAY$", r"^LOS$"],
    "admitType": [r"^TYPE_OF_ADMISSION$"],
    "drg":       [r"^MS_?DRG$"],
    "princDx":   [r"^PRINC_DIAG_CODE$", r"^PRINC(IPAL)?_DIAG"],
    "princProc": [r"^PRINC_SURG_PROC_CODE$", r"^PRINC_ICD\d+_PROC", r"^PRINC_PROC"],
    "poaPrinc":  [r"^POA_PRINC_DIAG_?CODE$"],              # present on admission code of the principal diagnosis
    "region":    [r"^PUBLIC_HEALTH_REGION$"],
    "country":   [r"^PAT_COUNTRY$"],
    "admitSource": [r"^SOURCE_OF_ADMISSION$"],
    "edFlag":    [r"^EMERGENCY_DEPT_FLAG$"],
    "admitDx":   [r"^ADMITTING_DIAGNOSIS$"],
}
notCopied = ("princDx", "princProc", "poaPrinc")   # handled separately, not copied as raw columns
requiredNames = ["recordId", "princDx"]

def mapColumns(cols):
    up = {c.upper().strip(): c for c in cols}
    found = {}
    for name, pats in columnPatterns.items():
        for p in pats:
            hit = [orig for u, orig in up.items() if re.search(p, u)]
            if hit:
                found[name] = hit[0]
                break
    othDx = sorted([orig for u, orig in up.items() if re.match(r"^OTH_DIAG_CODE_\d+$", u)],
                   key=lambda c: int(re.findall(r"\d+", c)[-1]))
    othProc = sorted([orig for u, orig in up.items() if re.match(r"^OTH_(SURG_)?PROC_CODE_\d+$|^OTH_ICD\d+_PROC_\d+$", u)],
                     key=lambda c: int(re.findall(r"\d+", c)[-1]))
    specUnit = sorted([orig for u, orig in up.items() if re.match(r"^SPEC_UNIT_\d+$", u)],
                      key=lambda c: int(re.findall(r"\d+", c)[-1]))
    othPoa = sorted([orig for u, orig in up.items() if re.match(r"^POA_OTH_DIAG_CODE_\d+$", u)],
                    key=lambda c: int(re.findall(r"\d+", c)[-1]))
    found["othDx"], found["othProc"], found["specUnit"], found["othPoa"] = othDx, othProc, specUnit, othPoa
    return found

baseMembers = pickBase(zipFiles[(2019, 1)])
baseMembers = [baseMembers] if isinstance(baseMembers, str) else baseMembers
if len(baseMembers) > 1:
    heads = {m: readHeader(zipFiles[(2019, 1)], m) for m in baseMembers}
    sameCols = len({tuple(h) for h in heads.values()}) == 1
    print("Several base files found:", baseMembers, "| same columns in all:", sameCols)
    assert sameCols, ("The base files have different columns, so they are probably split by column and must be joined on "
                      "the record ID, not stacked. Tell me the file names and columns and I will change the notebook.")
baseMember = baseMembers[0]
cols = readHeader(zipFiles[(2019, 1)], baseMember)
colMap = mapColumns(cols)
print(len(cols), "columns in", baseMember)
for k, v in colMap.items():
    print(f"  {k:10s} ->", v if not isinstance(v, list) else f"{len(v)} columns ({v[0]} ... {v[-1]})" if v else "NOT FOUND")

stillMissing = [n for n in requiredNames if n not in colMap]
assert not stillMissing and colMap["othDx"], (
    f"Could not find {stillMissing or 'other diagnosis columns'}. Compare the printed list with the User Manual "
    "and edit columnPatterns above.")
if not colMap["othProc"]:
    print("WARNING: no procedure columns found. The procedure-based SMM indicators (transfusion, ventilation, hysterectomy, "
          "tracheostomy, cardiac rhythm conversion) and procedure-based delivery flags will be skipped. Check the manual.")
print()
print("Specialty unit columns (code B = Obstetric Unit in the manual):", colMap["specUnit"])

## Step 3. Compare the column names across all eight quarters
The layout can change from one quarter to the next. This cell reads only the header of each file and stops if a quarter does not match 2019 Q1.

In [ ]:
baseCols = {}
for key, zp in zipFiles.items():
    mem = pickBase(zp)
    mem = mem if isinstance(mem, str) else mem[0]
    baseCols[key] = mapColumns(readHeader(zp, mem))
ref = baseCols[(2019, 1)]
for key, mp in baseCols.items():
    problems = [n for n in requiredNames if n not in mp]
    same = all(mp.get(n) == ref.get(n) for n in columnPatterns if n in ref) and len(mp["othDx"]) == len(ref["othDx"])
    print(f"{key[1]}Q{key[0]}: other dx columns = {len(mp['othDx'])}, other proc columns = {len(mp['othProc'])}, same names as 1Q2019 = {same}")
    assert not problems, f"{key}: missing {problems}"

## Step 4. Code lists
Delivery hospitalization, my definition: any of these.
- Diagnosis O80 (full-term delivery), O82 (cesarean delivery encounter), or any Z37 (outcome of delivery).
- A delivery MS-DRG: 765 to 768, 774, 775, 783 to 788, 796 to 798, 805 to 807.
- An ICD-10-PCS delivery procedure: 10D00Z0, 10D00Z1, 10D00Z2, 10E0XZZ.

This is a common administrative definition but it is my choice, not the CDC's, so cite it as such and compare the counts to CDC WONDER. Cesarean is O82, PCS 10D00Z0 to 10D00Z2, or cesarean DRGs 765, 766 and 783 to 788.

SMM: the 21 CDC indicators (the page lists them in this order, I numbered them). The CDC marks blood transfusion as "excluded, but could be examined separately", so my main outcome is smmNoTransfusion (indicators 1 to 8 and 10 to 21) and I also keep smmWithTransfusion.

ICD-10 version check (crosswalk). The code set changes every October 1, so these data use three versions: FY2019 (1Q to 3Q 2019), FY2020 (4Q2019 to 3Q2020) and FY2021 (4Q2020). I compared every code in the lists below with the official code files (NCHS ICD-10-CM 2019, 2020, 2021; CMS ICD-10-PCS 2019 and 2021):
- No listed code was deleted, and every listed procedure code is valid in all years.
- FY2020 added I26.93 and I26.94 (pulmonary embolism). The prefix I26 already catches them.
- FY2021 added 18 new sickle cell "with crisis" codes (for example D57.03, D57.43x, D57.45x). I added them to sickleCellCrisis. The "without crisis" codes D57.42 and D57.44 are left out, like D57.1.
- The new FY2021 transfusion codes (30230C0, 30233U2 and similar) are stem cell transplants, not blood products, so they are not added.
- The CDC list has no date per code, so this is my own harmonization. See docs/decisions.md.

PUDF diagnosis codes have no decimal point, so I remove dots and match by prefix. Prefix rule: I21.xx becomes I21, J96.0x becomes J960. Check this against the manual, because if the file keeps the dots then Step 6 will find nothing and the checks in Step 7 will show it.

In [ ]:
smmDx = {
 "ami": ["I21", "I22"],
 "aneurysm": ["I71", "I790"],
 "acuteRenalFailure": ["N17", "O904"],
 "ards": ["J80", "J951", "J952", "J953", "J9582", "J960", "J962", "J969", "R0603", "R092"],
 "amnioticEmbolism": ["O88112", "O88113", "O88119", "O8812", "O8813"],
 "cardiacArrest": ["I46", "I490"],
 "dic": ["D65", "D688", "D689", "O670", "O723"] + [f"O{a}{b}" for a in (45, 46) for b in
        ["002", "003", "009", "012", "013", "019", "022", "023", "029", "092", "093", "099"]],
 "eclampsia": ["O15"],
 "heartFailureSurgery": ["I97120", "I97121", "I97130", "I97131", "I97710", "I97711"],
 "puerperalCerebrovascular": ["A812", "G45", "G46", "G9349", "H340", "I60", "I61", "I62", "I6300", "I6301", "I631",
        "I632", "I633", "I634", "I635", "I636", "I638", "I639", "I65", "I66", "I67", "I68", "O2250", "O2252",
        "O2253", "I97810", "I97811", "I97820", "I97821", "O873"],
 "pulmonaryEdemaHeartFailure": ["I501", "I5020", "I5021", "I5023", "I5030", "I5031", "I5033", "I5040", "I5041",
        "I5043", "I50810", "I50811", "I50813", "I50814", "I5082", "I5083", "I5084", "I5089", "I509", "J810"],
 "anesthesiaComplications": [f"O29{a}{b}" for a in (11, 12, 19, 21, 29) for b in (2, 3, 9)]   # O29.112-O29.119 etc. (only these exist)
        + ["O740", "O741", "O742", "O743", "O890", "O891", "O892", "T882XXA", "T883XXA"],
 "sepsis": ["A327", "A40", "A41", "I76", "O85", "O8604", "R6520", "R6521", "T8112XA", "T8144XA"],
 "shock": ["O751", "R57", "T782XXA", "T8110XA", "T8111XA", "T8119XA", "T886XXA"],
 "sickleCellCrisis": ["D5700", "D5701", "D5702", "D57211", "D57212", "D57219", "D57411", "D57412", "D57419",
        "D57811", "D57812", "D57819"]
        # new "with crisis" codes from ICD-10-CM FY2021 (October 1, 2020, so 4Q2020 discharges)
        + ["D5703", "D5709", "D57213", "D57218", "D57413", "D57418", "D57431", "D57432", "D57433", "D57438", "D57439",
           "D57451", "D57452", "D57453", "D57458", "D57459", "D57813", "D57818"],
 "airThromboticEmbolism": ["I26", "T800XXA"] + [f"O88{a}{b}" for a in (0, 2, 3, 8) for b in ("12", "13", "19", "2", "3")],
}
transfusionSuffix = [f"{a}{b}" for b in "01" for a in "HKLMNPRT"]
smmPx = {
 "rhythmConversion": ["5A12012", "5A2204Z"],
 "hysterectomy": ["0UT90ZL", "0UT90ZZ", "0UT97ZL", "0UT97ZZ"],
 "tracheostomy": ["0B110F4", "0B113F4", "0B114F4"],
 "ventilation": ["5A1935Z", "5A1945Z", "5A1955Z"],
}
transfusionPx = [f"{p}{s}" for p in ("30230", "30233", "30240", "30243") for s in transfusionSuffix]

deliveryDx = ("O80", "O82", "Z37")
deliveryDrg = {765, 766, 767, 768, 774, 775, 783, 784, 785, 786, 787, 788, 796, 797, 798, 805, 806, 807}
cesareanDrg = {765, 766, 783, 784, 785, 786, 787, 788}
deliveryPx = ("10D00Z0", "10D00Z1", "10D00Z2", "10E0XZZ")
cesareanPx = ("10D00Z0", "10D00Z1", "10D00Z2")
print("diagnosis indicators:", len(smmDx), "| procedure indicators:", len(smmPx), "+ transfusion  (21 in total)")

## Step 4b. Comorbidity code lists
These flags are for adjustment step (b) in the project objectives. The list of conditions follows the obstetric comorbidity index idea (Bateman et al. 2013, ICD-10 version by Leonard et al. 2020), but the exact code prefixes are my own choice, so cite them as ours and check them against the paper before the final report.

Rules I used:
- A condition is flagged if any of the 25 diagnosis codes starts with one of its prefixes.
- No comorbidity prefix overlaps an SMM indicator code (checked against the ICD-10-CM FY2019, FY2020 and FY2021 code files). For example, eclampsia, DIC, acute renal failure, heart failure and sickle cell crisis are left out because they are part of the outcome.
- Two versions of every flag:
  - com_<name>: the code is anywhere on the record. This is the usual way comorbidity indexes are built.
  - comPoa_<name>: the code is present on admission (POA = Y or W). Use this for prediction, because a condition that starts during the stay is not known at admission. Hospitals that are exempt from POA reporting (for example critical access hospitals) have blank POA codes, so this version undercounts there.
- Severe preeclampsia and preterm birth can be diagnosed during the delivery stay. Use the POA version or a sensitivity check without them before treating them as baseline risk.
- Maternal age is not in this list because ageCat is already a feature.

In [ ]:
# comorbidity name -> ICD-10-CM prefixes (no dots). Chosen to NOT overlap the SMM indicator codes.
comorbidityDx = {
 "chronicHypertension":   ["O10", "O11", "I10", "I11", "I12", "I13", "I15", "I16"],
 "gestationalHypertension": ["O13", "O16"],
 "preeclampsia":          ["O140", "O149"],                     # without severe features or unspecified
 "severePreeclampsia":    ["O141", "O142"],                     # severe features or HELLP (eclampsia O15 is an SMM indicator)
 "pregestationalDiabetes": ["O240", "O241", "O243", "O248", "O249", "E08", "E09", "E10", "E11", "E13"],
 "gestationalDiabetes":   ["O244"],
 "obesity":               ["O9921", "E66", "Z683", "Z684"],      # Z68.30-Z68.45 = BMI 30 or more
 "bariatricSurgery":      ["O9984", "Z9884"],
 "asthma":                ["J45"],
 "substanceUse":          ["F10", "F11", "F12", "F13", "F14", "F15", "F16", "F18", "F19", "O9931", "O9932"],
 "tobaccoUse":            ["F17", "O9933", "Z720"],
 "mentalHealth":          ["F2", "F30", "F31", "F32", "F33", "F34", "F39", "F40", "F41", "F42", "F43", "F44", "F45",
                           "F48", "F53", "O9934"],
 "priorCesarean":         ["O3421"],
 "multipleGestation":     ["O30", "O31", "Z372", "Z373", "Z374", "Z375", "Z376", "Z377"],
 "placentaPrevia":        ["O44"],
 "placentaAccreta":       ["O432"],
 "placentalAbruption":    ["O458", "O459"],             # O45.0 (with coagulation defect) is in the SMM DIC indicator
 "pretermBirth":          ["O601"],                             # preterm labor with preterm delivery
 "anemia":                ["O990", "D50", "D51", "D52", "D53", "D55", "D56", "D58", "D59", "D60", "D61", "D63", "D64"],
 "sickleCellDisease":     ["D571", "D5720", "D5740", "D5742", "D5744", "D5780"],   # without crisis (crisis is SMM)
 "bleedingDisorder":      ["D66", "D67", "D680", "D681", "D682", "D6831", "D6832", "D684", "D69"],  # not D65, D68.8, D68.9 (SMM DIC)
 "cardiacDisease":        ["I05", "I06", "I07", "I08", "I09", "I20", "I25", "I34", "I35", "I36", "I37", "I42", "I44",
                           "I45", "I47", "I48", "Q20", "Q21", "Q22", "Q23", "Q24", "Z95"],   # not I21, I46, I49.0, I50 (SMM)
 "pulmonaryHypertension": ["I27"],
 "chronicRenalDisease":   ["N18", "N03", "N04", "N11", "N25", "O102", "O103", "O2683", "Z940", "Z992"],  # not N17 (SMM)
 "autoimmuneDisease":     ["M05", "M06", "M30", "M31", "M32", "M33", "M34", "M35"],
 "thyroidDisease":        ["E00", "E01", "E02", "E03", "E04", "E05", "E06", "E07"],
 "neuromuscularDisease":  ["G35", "G40", "G70", "G71"],
 "hiv":                   ["B20", "Z21", "O987"],
 "liverOrBowelDisease":   ["K50", "K51", "K70", "K71", "K72", "K73", "K74", "K75", "K76"],
}

print(len(comorbidityDx), "comorbidity groups")

## Step 5. Process one quarter
For each quarter the notebook reads the base file in chunks of 100,000 rows and drops the rows that are not deliveries right away. This keeps memory use small. It removes the record IDs from the 2020 update files (Q2 and Q3). Then it flags SMM.

In [ ]:
def cleanCodes(frame):
    return frame.fillna("").astype(str).apply(lambda s: s.str.upper().str.replace(".", "", regex=False).str.strip())

def anyStarts(frame, prefixes):
    prefixes = tuple(prefixes)
    hit = np.zeros(len(frame), dtype=bool)
    for c in frame.columns:
        hit |= frame[c].str.startswith(prefixes).to_numpy()
    return hit

def loadExcludeIds(spec):
    if spec is None:
        return set()
    path, sheet = spec if isinstance(spec, tuple) else (spec, 0)
    path = Path(path)
    assert path.exists(), f"Exclude file not found: {path}"
    x = pd.read_excel(path, sheet_name=sheet, header=None, dtype=str)
    # the RECORD_ID header is not always in the first row, so find the cell first
    hits = [(r, c) for r in range(len(x)) for c in range(x.shape[1])
            if str(x.iat[r, c]).upper().replace(" ", "").replace("_", "") == "RECORDID"]
    assert hits, f"No RECORD_ID header in {path.name}, sheet {sheet}"
    r, c = hits[0]
    vals = x.iloc[r + 1:, c].dropna().astype(str).str.strip()
    ids = set(vals[vals.str.fullmatch(r"\d+")])
    print(f"  loaded {len(ids):,} record IDs to exclude from {path.name}, sheet {sheet}")
    return ids

ageWarned = {"done": False}

def anyStartsPoa(dx, poa, prefixes):
    """Like anyStarts, but returns two arrays: the code is anywhere, and the code is present on admission (POA Y or W)."""
    prefixes = tuple(prefixes)
    anywhere = np.zeros(len(dx), dtype=bool)
    onAdmission = np.zeros(len(dx), dtype=bool)
    for j, c in enumerate(dx.columns):
        hit = dx[c].str.startswith(prefixes).to_numpy()
        anywhere |= hit
        onAdmission |= hit & poa[:, j]
    return anywhere, onAdmission

def loadFacility(zipPath):
    """Facility type file (one row per hospital per quarter): teaching status, POA reporting status."""
    with zipfile.ZipFile(zipPath) as z:
        mem = [m for m in z.namelist() if "facility_type" in Path(m).name.lower()
               and "__macosx" not in m.lower() and not Path(m).name.startswith("._")]
        assert len(mem) == 1, f"Expected one facility type file in {zipPath.name}, found {mem}"
        with z.open(mem[0]) as f:
            fac = pd.read_csv(f, sep="\t", dtype=str, encoding="latin-1")
    fac.columns = [c.strip().upper() for c in fac.columns]
    fac = fac[["THCIC_ID", "FAC_TEACHING_IND", "FAC_ACUTE_CARE_IND", "POA_PROVIDER_INDICATOR"]].copy()
    fac = fac.apply(lambda s: s.fillna("").str.strip())
    return fac.rename(columns={"THCIC_ID": "hospId", "FAC_TEACHING_IND": "facTeaching",
                               "FAC_ACUTE_CARE_IND": "facAcuteCare", "POA_PROVIDER_INDICATOR": "facPoaReporting"})

def processQuarter(year, q):
    zp = zipFiles[(year, q)]
    mem = pickBase(zp)
    mems = [mem] if isinstance(mem, str) else mem
    mp = mapColumns(readHeader(zp, mems[0]))
    dxCols = [mp["princDx"]] + mp["othDx"]
    pxCols = ([mp["princProc"]] if "princProc" in mp else []) + mp["othProc"]
    specCols = mp["specUnit"]
    poaCols = ([mp["poaPrinc"]] if "poaPrinc" in mp else []) + mp["othPoa"]
    assert len(poaCols) == len(dxCols), f"{q}Q{year}: {len(poaCols)} POA columns for {len(dxCols)} diagnosis columns"
    keep = [mp[k] for k in columnPatterns if k in mp and k not in notCopied]
    excludeIds = loadExcludeIds(excludeIdFiles.get((year, q)))
    out, nRead, nExcluded = [], 0, 0
    for member in mems:
        with zipfile.ZipFile(zp) as z, z.open(member) as f:
            for chunk in pd.read_csv(f, sep="\t", dtype=str, encoding="latin-1", chunksize=chunkSize,
                                     usecols=lambda c: c in set(keep + dxCols + pxCols + specCols + poaCols)):
                nRead += len(chunk)
                if excludeIds:
                    drop = chunk[mp["recordId"]].astype(str).str.strip().isin(excludeIds)
                    nExcluded += int(drop.sum())
                    chunk = chunk[~drop]
                dx = cleanCodes(chunk[dxCols])
                px = cleanCodes(chunk[pxCols]) if pxCols else None
                drgNum = pd.to_numeric(chunk[mp["drg"]], errors="coerce") if "drg" in mp else pd.Series(np.nan, index=chunk.index)
                isDx = anyStarts(dx, deliveryDx)
                isDrg = drgNum.isin(deliveryDrg).to_numpy()
                isPx = anyStarts(px, deliveryPx) if px is not None else np.zeros(len(chunk), dtype=bool)
                isDelivery = isDx | isDrg | isPx
                if not isDelivery.any():
                    continue
                c2 = chunk[isDelivery].copy()
                dx, drgNum = dx[isDelivery], drgNum[isDelivery]
                px = px[isDelivery] if px is not None else None
                res = pd.DataFrame({"year": year, "quarter": q}, index=c2.index)
                for k in columnPatterns:
                    if k in mp and k not in notCopied:
                        res[k] = c2[mp[k]].to_numpy()
                res["princDx"] = dx[mp["princDx"]].to_numpy()
                res["delivDx"], res["delivDrg"] = isDx[isDelivery], isDrg[isDelivery]
                res["delivPx"] = isPx[isDelivery]
                cesDx = anyStarts(dx, ("O82",))
                cesPx = anyStarts(px, cesareanPx) if px is not None else np.zeros(len(c2), dtype=bool)
                res["cesarean"] = cesDx | cesPx | drgNum.isin(cesareanDrg).to_numpy()
                for name, pre in smmDx.items():
                    res["smm_" + name] = anyStarts(dx, pre)
                for name, pre in smmPx.items():
                    res["smm_" + name] = anyStarts(px, pre) if px is not None else False
                res["smm_transfusion"] = anyStarts(px, transfusionPx) if px is not None else False
                res["nDxCodes"] = (dx != "").sum(axis=1).to_numpy()
                spec = c2[specCols].fillna("").apply(lambda s: s.str.strip().str.upper()) if specCols else None
                res["obUnit"] = (spec == "B").any(axis=1).to_numpy() if spec is not None else False
                # comorbidities: anywhere on the record, and present on admission
                poaCodes = c2[poaCols].fillna("").apply(lambda s: s.str.strip().str.upper()).to_numpy()
                poaYes = np.isin(poaCodes, ["Y", "W"])
                comFlags = {}
                for name, pre in comorbidityDx.items():
                    comFlags["com_" + name], comFlags["comPoa_" + name] = anyStartsPoa(dx, poaYes, pre)
                hasCode = (dx != "").to_numpy()
                comFlags["poaAllBlank"] = ~((poaCodes != "") & hasCode).any(axis=1)   # no POA code on any diagnosis
                res = pd.concat([res, pd.DataFrame(comFlags, index=res.index)], axis=1)   # add all at once (faster)
                out.append(res)
    df = pd.concat(out, ignore_index=True) if out else pd.DataFrame()
    df["hospId"] = df["hospId"].fillna("").astype(str).str.strip()
    nBefore = len(df)
    df = df.merge(loadFacility(zp), on="hospId", how="left")
    assert len(df) == nBefore, "facility file has duplicate hospital IDs"
    print(f"{q}Q{year}: read {nRead:,} rows, excluded {nExcluded:,}, kept {len(df):,} delivery rows")
    return df

## Step 6. Run all eight quarters, clean, and save
After combining the quarters this cell cleans the patient fields with the code tables from the User Manual.
- Age: PAT_AGE is an age group code, not years. The CDC range of 12 to 55 cannot be cut exactly, so I keep codes 04 to 13 (10 to 54 years). Codes 22 to 24 are wider groups that DSHS uses for patients with HIV or drug or alcohol codes (their sex, ZIP and county are also masked). I keep them as their own groups, because dropping them would remove a higher risk population.
- Race and ethnicity: I combine them into one variable, raceEth. Hispanic of any race is "Hispanic", and the others are non-Hispanic (NH) by race. This matters because about 19% of deliveries have race 5 (Other) and most of them are Hispanic.
- Suppression: the manual says that if a hospital has fewer than 10 patients of a race, the race is changed to 5 (Other) and the ethnicity is left blank. So race 5 with a blank ethnicity is the suppression pattern. These rows get the category "Suppressed" and the flag raceSuppressed. Invalid or blank codes get "Unknown". No imputation (treated as missing not at random).
- Death: PAT_STATUS 20, 40, 41 or 42 (expired).
- Obstetric unit: any SPEC_UNIT column equal to B. Kept as a flag only, not used to define the cohort.
- Payer: the 23 codes grouped into 7 groups.
- Residence (PAT_STATE, User Manual): TX = Texas; AR, LA, NM, OK and ZZ (all other states and territories) = other US state; FC and XX = foreign country. A back quote or blank is unknown (masked HIV or drug or alcohol records).
- Public Health Region (PUBLIC_HEALTH_REGION, 1 to 11) of the patient's address, labeled with the main cities. Blank for non-Texas residents.
- Admission type: emergency, urgent, elective, other or unknown. Source of admission: transferIn = transfer from another hospital (code 4). Emergency department flag: edVisit.
- Hospital teaching status from the facility type file: COTH member (A), other teaching (X), non-teaching. Hospitals with a masked ID are "Unknown".
- Comorbidity counts: nComorbidities (anywhere) and nComorbiditiesPoa (present on admission).

In [ ]:
parts = [processQuarter(y, q) for y in years for q in quarters]
cohort = pd.concat(parts, ignore_index=True)

smmCols = [c for c in cohort.columns if c.startswith("smm_") and c != "smm_transfusion"]
cohort["smmCount"] = cohort[smmCols].sum(axis=1)
cohort["smmNoTransfusion"] = cohort["smmCount"] > 0
cohort["smmWithTransfusion"] = cohort["smmNoTransfusion"] | cohort["smm_transfusion"]

def clean(s):
    return cohort[s].fillna("").astype(str).str.strip()

# ---- age (User Manual, Field 22, PAT_AGE) ----
ageGroupLabels = {"04": "10-14", "05": "15-17", "06": "18-19", "07": "20-24", "08": "25-29", "09": "30-34",
                  "10": "35-39", "11": "40-44", "12": "45-49", "13": "50-54",
                  "22": "0-17 (masked)", "23": "18-44 (masked)", "24": "45-64 (masked)"}
ageCatMap = {"04": "<20", "05": "<20", "06": "<20", "22": "<20", "07": "20-24", "08": "25-29", "09": "30-34",
             "10": "35-39", "11": "40+", "12": "40+", "13": "40+", "24": "40+", "23": "18-44 (masked)"}
ageCode = clean("age")
keepAge = ageCode.isin(ageGroupLabels)
print("Rows dropped by the age filter, by PAT_AGE code:", ageCode[~keepAge].value_counts().to_dict())
cohort = cohort[keepAge].copy()
ageCode = ageCode[keepAge]
cohort["ageGroup"] = ageCode.map(ageGroupLabels)
cohort["ageCat"] = ageCode.map(ageCatMap)
cohort["ageMasked"] = ageCode.isin(["22", "23", "24"])

# ---- race and ethnicity ----
race, eth = clean("race"), clean("ethnicity")
cohort["raceLabel"] = race.map(raceLabels).fillna("Unknown or invalid")
cohort["ethnicityLabel"] = eth.map(ethnicityLabels).fillna("Blank or invalid")
cohort["raceSuppressed"] = (race == "5") & (eth == "")
nhMap = {"4": "NH White", "3": "NH Black", "2": "NH Asian or Pacific Islander", "1": "NH American Indian", "5": "NH Other"}
raceEth = pd.Series("Unknown", index=cohort.index, dtype=object)
raceEth[eth == "2"] = race[eth == "2"].map(nhMap)
raceEth = raceEth.fillna("Unknown")
raceEth[eth == "1"] = "Hispanic"
raceEth[cohort["raceSuppressed"]] = "Suppressed"
raceEthOrder = ["NH White", "Hispanic", "NH Black", "NH Asian or Pacific Islander", "NH American Indian",
                "NH Other", "Suppressed", "Unknown"]
cohort["raceEth"] = pd.Categorical(raceEth, categories=raceEthOrder)
cohort["raceEthMissing"] = cohort["raceEth"].isin(["Suppressed", "Unknown"])

# ---- discharge status, payer, hospital ----
status = clean("status")
cohort["diedInHospital"] = status.isin(["20", "40", "41", "42"])
cohort["toHospice"] = status.isin(["50", "51"])
payerGroupMap = {"MC": "Medicaid",
                 "12": "Private", "13": "Private", "14": "Private", "15": "Private", "BL": "Private", "HM": "Private", "CI": "Private",
                 "MA": "Medicare", "MB": "Medicare", "16": "Medicare",
                 "ZZ": "Self-pay or charity", "09": "Self-pay or charity",
                 "CH": "Other government", "VA": "Other government", "OF": "Other government", "TV": "Other government",
                 "10": "Other", "11": "Other", "AM": "Other", "LM": "Other", "LI": "Other", "WC": "Other", "DS": "Other"}
cohort["payerGroup"] = clean("payer").map(payerGroupMap).fillna("Unknown")
cohort["hospMasked"] = clean("hospId").isin(["999998", "999999"])   # small or masked hospitals (manual)

# ---- residence (PAT_STATE, User Manual) ----
st = clean("state")
cohort["residence"] = np.select(
    [st == "TX", st.isin(["AR", "LA", "NM", "OK", "ZZ"]), st.isin(["FC", "XX"])],
    ["Texas", "Other US state", "Foreign country"], "Unknown")

# ---- Public Health Region of the patient's address ----
regionLabels = {"1": "1 Lubbock/Amarillo", "2": "2 Abilene/Wichita Falls", "3": "3 Dallas/Fort Worth", "4": "4 Tyler/Texarkana",
                "5": "5 Beaumont/Lufkin", "6": "6 Houston", "7": "7 Austin/Waco", "8": "8 San Antonio", "9": "9 Midland/Odessa",
                "10": "10 El Paso", "11": "11 Corpus Christi/Valley/Laredo"}
regionCode = clean("region").str.lstrip("0")
cohort["region"] = regionCode.map(regionLabels).fillna("Unknown or outside Texas")

# ---- admission type, source, emergency department ----
cohort["admitGroup"] = clean("admitType").map({"1": "Emergency", "2": "Urgent", "3": "Elective"}).fillna("Other or unknown")
cohort["transferIn"] = clean("admitSource") == "4"
cohort["edVisit"] = clean("edFlag") == "Y"

# ---- hospital teaching status (facility type file) ----
cohort["teaching"] = cohort["facTeaching"].map({"A": "COTH member", "X": "Other teaching", "": "Non-teaching"}).fillna("Unknown")

# ---- comorbidity counts ----
comCols = ["com_" + n for n in comorbidityDx]
comPoaCols = ["comPoa_" + n for n in comorbidityDx]
cohort["nComorbidities"] = cohort[comCols].sum(axis=1)
cohort["nComorbiditiesPoa"] = cohort[comPoaCols].sum(axis=1)

outPath = outDir / "delivery_cohort_2019_2020.parquet"
try:
    cohort.to_parquet(outPath, index=False)
except Exception as e:
    outPath = outDir / "delivery_cohort_2019_2020.csv.gz"
    cohort.to_csv(outPath, index=False)
print("Saved", len(cohort), "rows to", outPath)

## Step 7. Checks
What to look for:
- Deliveries per year should be near Texas births, roughly 370,000 to 390,000 a year. If you are far below, the delivery filter or the file is wrong. A small gap is expected because some births happen outside hospitals and the public file leaves some hospitals out.
- All three delivery criteria should agree for most rows. If only the DRG flag fires, the diagnosis codes may still have dots in them.
- The SMM rate is usually somewhat under 1 percent without transfusion and around 2 percent with transfusion. Far outside this range means a code problem.
- Deaths in the hospital are rare (tens per year), too few to model.

In [ ]:
print("Delivery rows per year and quarter")
display(cohort.groupby(["year", "quarter"]).size().unstack())
print()
crit = cohort[["delivDx", "delivDrg", "delivPx"]].astype(int)
print("How many rows meet each delivery criterion (and how many meet all three):")
print(crit.sum().to_dict(), "| all three:", int((crit.sum(axis=1) == 3).sum()))
print("Rows with a record ID repeated inside a quarter:",
      int(cohort.duplicated(["year", "quarter", "recordId"]).sum()) if "recordId" in cohort else "recordId not kept")
print()
print(f"Cesarean share: {cohort['cesarean'].mean():.3f}")
print(f"SMM rate without transfusion: {cohort['smmNoTransfusion'].mean()*1000:.1f} per 1,000 deliveries")
print(f"SMM rate with transfusion    : {cohort['smmWithTransfusion'].mean()*1000:.1f} per 1,000 deliveries")
print()
byIndicator = (cohort[[c for c in cohort.columns if c.startswith("smm_")]].sum().sort_values(ascending=False)
               .rename("deliveries with indicator").to_frame())
display(byIndicator)
print()
print()
print("Race and ethnicity (combined):")
display(cohort["raceEth"].value_counts().rename("deliveries").to_frame()
        .assign(share=lambda d: (d["deliveries"] / len(cohort)).round(4)))
print("Suppressed race (race 5 with blank ethnicity):", int(cohort["raceSuppressed"].sum()))
print("Race x ethnicity codes:")
display(pd.crosstab(cohort["raceLabel"], cohort["ethnicityLabel"]))
print("Age groups:", cohort["ageGroup"].value_counts().to_dict())
print("Payer groups:", cohort["payerGroup"].value_counts().to_dict())
print("Died in hospital:", int(cohort["diedInHospital"].sum()), "| discharged to hospice:", int(cohort["toHospice"].sum()))
print(f"Obstetric unit flag (SPEC_UNIT = B): {cohort['obUnit'].mean():.3f}")
print(f"Masked hospital ID (999998 or 999999): {cohort['hospMasked'].mean():.4f}")
print()
for c in ("race", "ethnicity", "county", "payer", "status", "age"):
    if c in cohort:
        print(f"{c:10s} missing or blank: {(cohort[c].isna() | (cohort[c].astype(str).str.strip() == '')).mean():.3f}   "
              f"distinct values: {cohort[c].nunique()}")

## Step 7b. Checks for the comorbidity flags and the new fields
What to look for:
- Prevalence should be in a plausible range for US deliveries. For example prior cesarean around 15 to 20%, gestational diabetes around 6 to 10%, chronic hypertension around 2 to 4%, asthma a few percent.
- The POA version should be a bit lower than the anywhere version. A big gap means the condition is often coded during the stay, or many records come from hospitals that do not report POA.
- Each comorbidity should raise the SMM rate. A rate ratio far below 1 suggests a code problem.
- Facility file: almost every hospital ID should match.

In [ ]:
comTable = pd.DataFrame({
    "anywherePct": cohort[comCols].mean().to_numpy() * 100,
    "poaPct": cohort[comPoaCols].mean().to_numpy() * 100,
}, index=list(comorbidityDx))
smmWith = [cohort.loc[cohort[c], "smmNoTransfusion"].mean() * 1000 for c in comCols]
smmWithout = [cohort.loc[~cohort[c], "smmNoTransfusion"].mean() * 1000 for c in comCols]
comTable["smmPer1000With"], comTable["smmPer1000Without"] = smmWith, smmWithout
comTable["rateRatio"] = comTable["smmPer1000With"] / comTable["smmPer1000Without"]
comTable["deliveries"] = cohort[comCols].sum().to_numpy()
display(comTable.sort_values("anywherePct", ascending=False).round(2))

print("Number of comorbidities per delivery (anywhere):")
display(cohort["nComorbidities"].clip(upper=5).value_counts().sort_index().rename("deliveries").to_frame())
print("SMM per 1,000 by number of comorbidities (5 = 5 or more):")
display((cohort.groupby(cohort["nComorbidities"].clip(upper=5))["smmNoTransfusion"].mean() * 1000).round(2).to_frame())

print(f"Deliveries with no POA code on any diagnosis: {cohort['poaAllBlank'].mean():.3f}")
print("Facility POA reporting status (R required, M mixed, X exempt, blank = not matched):")
display(cohort["facPoaReporting"].fillna("not matched").value_counts().to_frame())
print("Share with no POA codes by facility POA status:")
display(cohort.groupby(cohort["facPoaReporting"].fillna("not matched"))["poaAllBlank"].mean().round(3).to_frame())

print("Comorbidity prevalence (%) by race and ethnicity, anywhere version:")
mainGroups = ["NH White", "Hispanic", "NH Black", "NH Asian or Pacific Islander", "NH Other"]
display((cohort.groupby("raceEth", observed=True)[comCols].mean().T * 100).rename(index=lambda c: c[4:])[mainGroups].round(1))

print("Residence:", cohort["residence"].value_counts().to_dict())
print("Region by residence (Texas residents should almost never be Unknown):")
display(pd.crosstab(cohort["region"], cohort["residence"]))
print("Admission group:", cohort["admitGroup"].value_counts().to_dict())
print(f"Transfer in from another hospital: {cohort['transferIn'].mean():.3f}, emergency department flag: {cohort['edVisit'].mean():.3f}")
print("Teaching status:", cohort["teaching"].value_counts().to_dict())

## Step 8. Quick plots

In [ ]:
byQ = cohort.groupby(["year", "quarter"]).agg(deliveries=("cesarean", "size"), cesarean=("cesarean", "mean"),
                                               smm=("smmNoTransfusion", "mean")).reset_index()
byQ["label"] = byQ["quarter"].astype(str) + "Q" + byQ["year"].astype(str)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].bar(byQ["label"], byQ["deliveries"], color=attBlue)
ax[0].set_title("Delivery hospitalizations per quarter"); ax[0].tick_params(axis="x", rotation=45)
ax[1].plot(byQ["label"], byQ["cesarean"] * 100, marker="o", color=attOrange, label="cesarean share (%)")
ax[1].plot(byQ["label"], byQ["smm"] * 100, marker="o", color=attBlue, label="SMM without transfusion (%)")
ax[1].set_title("Cesarean and SMM rates"); ax[1].tick_params(axis="x", rotation=45); ax[1].legend(frameon=False)
for a in ax:
    a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## Step 9. SMM by race and ethnicity
Rates per 1,000 deliveries with 95% Wilson confidence intervals, and the rate ratio compared with NH White.
- Main outcome: SMM without transfusion. Sensitivity: SMM with transfusion, and 2019 only (2020 is a COVID year).
- Groups with fewer than 20 SMM cases are marked. Their rates are unstable, so merge them or report them with the wide interval.
- Before you put counts in a report, check the Data Use Agreement for its rule on small cells.

In [ ]:
def wilson(k, n, z=1.96):
    k, n = np.asarray(k, float), np.asarray(n, float)
    p = k / n
    den = 1 + z**2 / n
    mid = (p + z**2 / (2 * n)) / den
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return mid - half, mid + half

def rateTable(frame, outcome, label):
    g = frame.groupby("raceEth", observed=True)[outcome].agg(cases="sum", deliveries="size")
    lo, hi = wilson(g["cases"], g["deliveries"])
    g["per1000"] = g["cases"] / g["deliveries"] * 1000
    g["ciLow"], g["ciHigh"] = lo * 1000, hi * 1000
    g["rateRatio"] = g["per1000"] / g.loc["NH White", "per1000"]
    g["fewCases"] = g["cases"] < 20
    g.columns = pd.MultiIndex.from_product([[label], g.columns])
    return g

smmByRace = pd.concat([rateTable(cohort, "smmNoTransfusion", "SMM no transfusion, 2019-2020"),
                       rateTable(cohort, "smmWithTransfusion", "SMM with transfusion, 2019-2020"),
                       rateTable(cohort[cohort["year"] == 2019], "smmNoTransfusion", "SMM no transfusion, 2019 only")], axis=1)
display(smmByRace.round(2))
smmByRace.to_csv(outDir / "smm_by_race_ethnicity.csv")

main = smmByRace["SMM no transfusion, 2019-2020"]
only2019 = smmByRace["SMM no transfusion, 2019 only"]
yPos = np.arange(len(main))
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(yPos + 0.2, main["per1000"], height=0.4, color=attBlue, label="2019-2020",
        xerr=[main["per1000"] - main["ciLow"], main["ciHigh"] - main["per1000"]], error_kw={"ecolor": attGray})
ax.barh(yPos - 0.2, only2019["per1000"].reindex(main.index), height=0.4, color=attOrange, label="2019 only",
        xerr=[(only2019["per1000"] - only2019["ciLow"]).reindex(main.index),
              (only2019["ciHigh"] - only2019["per1000"]).reindex(main.index)], error_kw={"ecolor": attGray})
ax.set_yticks(yPos, main.index); ax.invert_yaxis()
ax.set_xlabel("SMM without transfusion per 1,000 deliveries (95% CI)")
ax.set_title("SMM by race and ethnicity, Texas PUDF"); ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

print("In-hospital deaths by race and ethnicity (too few to model, descriptive only):")
display(cohort.groupby("raceEth", observed=True)["diedInHospital"].sum().rename("deaths").to_frame())

## Step 10. Encode the features for modeling
- Features: raceEth, ageCat, payerGroup, admitGroup (admission type), teaching (hospital teaching status), and year 2020. These are known at admission.
- Comorbidities are saved as 0/1 columns in both versions (com_ anywhere, comPoa_ present on admission), so each model can pick one. Use com_ for the disparity adjustment and comPoa_ for prediction.
- Saved as information only, not as features: region, residence, transferIn, edVisit. A transfer in can happen because the complication already started, so it is a mediator for the disparity question. Add it to the prediction model only if the team agrees.
- Not used as features: length of stay, discharge status, the number of diagnosis codes, and the SMM indicators. They are measured during or after the stay and partly caused by SMM, so using them would leak the outcome. Cesarean is also left out by default because it happens during the same stay. Add it only if your question needs it.
- Dummy coding with a reference group (NH White, 25-29, Private, Elective, Non-teaching), for regression. Set dropReference = False for full one-hot coding (trees and other models that do not need a reference).
- Missing race and ethnicity stay as their own categories (Suppressed, Unknown), with no imputation. The raceSuppressed flag is kept in the file but not added to the model matrix, because it is identical to the Suppressed dummy and would make the regression collinear.

In [ ]:
dropReference = True
references = {"raceEth": "NH White", "ageCat": "25-29", "payerGroup": "Private", "admitGroup": "Elective", "teaching": "Non-teaching"}
ageCatOrder = ["<20", "20-24", "25-29", "30-34", "35-39", "40+", "18-44 (masked)"]
payerOrder = ["Private", "Medicaid", "Medicare", "Self-pay or charity", "Other government", "Other", "Unknown"]
admitOrder = ["Elective", "Emergency", "Urgent", "Other or unknown"]
teachingOrder = ["Non-teaching", "COTH member", "Other teaching", "Unknown"]
orders = {"raceEth": raceEthOrder, "ageCat": ageCatOrder, "payerGroup": payerOrder, "admitGroup": admitOrder, "teaching": teachingOrder}

feat = pd.DataFrame(index=cohort.index)
for col, order in orders.items():
    ref = references[col]
    levels = [ref] + [x for x in order if x != ref]
    feat[col] = pd.Categorical(cohort[col].astype(str), categories=levels)
assert not feat.isna().any().any(), "A value is missing from the category lists above"

X = pd.get_dummies(feat, prefix_sep="_", drop_first=dropReference, dtype=int)
X.columns = [re.sub(r"[^0-9A-Za-z]+", "_", c.replace("<", "under").replace("+", "plus")).strip("_") for c in X.columns]
X["year2020"] = (cohort["year"] == 2020).astype(int)

idCols = ["recordId", "hospId", "year", "quarter"]
outcomeCols = ["smmNoTransfusion", "smmWithTransfusion", "diedInHospital"]
infoCols = ["raceEth", "raceSuppressed", "raceEthMissing", "ageMasked", "cesarean", "obUnit", "hospMasked",
            "region", "residence", "transferIn", "edVisit", "poaAllBlank"]
comorbidityCols = comCols + comPoaCols + ["nComorbidities", "nComorbiditiesPoa"]
modelData = pd.concat([cohort[idCols + outcomeCols + infoCols].reset_index(drop=True),
                       X.reset_index(drop=True),
                       cohort[comorbidityCols].astype(int).reset_index(drop=True)], axis=1)
modelData[outcomeCols] = modelData[outcomeCols].astype(int)

print(X.shape[1], "encoded feature columns:", list(X.columns))
print(len(comorbidityCols), "comorbidity columns (com_ anywhere, comPoa_ present on admission, and the two counts)")
print("References (left out):", references if dropReference else "none, full one-hot")
modelPath = outDir / "model_data_2019_2020.parquet"
modelData.to_parquet(modelPath, index=False)
print("Saved", modelData.shape, "to", modelPath)

## Notes and limits
Checked against the User Manual (code tables are the same in all 8 quarters):
- Age is a group code, so the cohort is 10 to 54 years plus the masked groups, not exactly 12 to 55.
- Race 5 with a blank ethnicity is treated as suppressed. A patient whose race really is Other and whose ethnicity was left blank looks the same, so this flag slightly overcounts suppression.
- Procedure codes are in the base file (principal plus 24 others), so all 21 SMM indicators are covered.
- The obstetric unit flag exists (SPEC_UNIT = B). It is kept as a flag only.
- The county column is masked when a county has fewer than five discharges in a quarter, and for the masked HIV and drug or alcohol records.

Still open:
- Comorbidity prefixes are my own list (see Step 4b), not an official index. The weighted obstetric comorbidity index of Leonard et al. 2020 can be built from these flags if the team copies the weights from the paper.
- POA codes are blank for hospitals exempt from POA reporting, so comPoa_ flags undercount there.
- The SMM flags still count conditions at any point in the stay (no POA filter). The comorbidity flags have a POA version.
- Cite DSHS (Texas Health Care Information Collection, Texas Inpatient Public Use Data File) in every write-up.